In [1]:
import os

for name in os.listdir("/kaggle/input/datasets/zlemglsmklkaya"):
    print(name)

caai-revision-splits
caai-models-nb1-complete


In [2]:
import os, json
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest

SEED = 42
np.random.seed(SEED)

SPLITS_ROOT = "/kaggle/input/datasets/zlemglsmklkaya/caai-revision-splits"
OUTPUT_DIR = "/kaggle/working/models"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Credit card
cc_dir = os.path.join(SPLITS_ROOT, "credit_card")
cc_train = pd.read_parquet(os.path.join(cc_dir, "train.parquet"))
cc_id    = pd.read_parquet(os.path.join(cc_dir, "id.parquet"))
cc_id2   = pd.read_parquet(os.path.join(cc_dir, "id2.parquet"))
cc_ood   = pd.read_parquet(os.path.join(cc_dir, "ood.parquet"))
with open(os.path.join(cc_dir, "config.json")) as f:
    cc_config = json.load(f)
CC_FEATURES = cc_config["model_features"]
print(f"Credit card: Train={cc_train.shape}, ID={cc_id.shape}, ID2={cc_id2.shape}, OOD={cc_ood.shape}")

# CIC-IDS2017, both scenarios
cic_scenarios = {}
for scen in ["scenario_T", "scenario_A"]:
    d = os.path.join(SPLITS_ROOT, "cic_ids2017", scen)
    cic_scenarios[scen] = {
        "train": pd.read_parquet(os.path.join(d, "train.parquet")),
        "id":    pd.read_parquet(os.path.join(d, "id.parquet")),
        "id2":   pd.read_parquet(os.path.join(d, "id2.parquet")),
        "ood":   pd.read_parquet(os.path.join(d, "ood.parquet")),
    }
with open(os.path.join(SPLITS_ROOT, "cic_ids2017", "config.json")) as f:
    cic_config = json.load(f)
CIC_FEATURES = cic_config["model_features"]
print(f"CIC-IDS2017 features ({len(CIC_FEATURES)})")
for scen, d in cic_scenarios.items():
    print(f"{scen}: Train={d['train'].shape}, ID={d['id'].shape}, ID2={d['id2'].shape}, OOD={d['ood'].shape}")

Credit card: Train=(100774, 31), ID=(21731, 31), ID2=(21731, 31), OOD=(139490, 31)
CIC-IDS2017 features (67)
scenario_T: Train=(910016, 82), ID=(296457, 82), ID2=(296457, 82), OOD=(1017868, 82)
scenario_A: Train=(1190196, 82), ID=(357584, 82), ID2=(357585, 82), OOD=(615433, 82)


In [3]:
IF_CONFIG = {
    "n_estimators": 100,
    "max_samples": "auto",
    "contamination": "auto",
    "random_state": SEED,
    "n_jobs": -1,
}
print(json.dumps(IF_CONFIG, indent=2))


def fit_isolation_forest(train_df, feature_cols, config):
    """Fit on the FULL Train set (benign-only) — no val split needed,
    since Isolation Forest has no early-stopping / overfitting-prevention mechanism
    that a held-out set would serve."""
    model = IsolationForest(**config)
    model.fit(train_df[feature_cols].values)
    return model


def compute_if_score(model, df, feature_cols):
    """sklearn's score_samples() returns LOWER = more anomalous — opposite
    convention from AE/VAE (higher = more anomalous). Flip the sign here,
    once, so every downstream notebook can assume higher-is-worse consistently."""
    raw_scores = model.score_samples(df[feature_cols].values)
    return -raw_scores


def score_split_if(model, df, feature_cols, split_name):
    scores = compute_if_score(model, df, feature_cols)
    out = pd.DataFrame({"row_id": np.arange(len(df)), "Class": df["Class"].values, "if_score": scores})
    benign = out[out.Class == 0].if_score
    anomaly = out[out.Class == 1].if_score
    print(f"{split_name}: benign mean={benign.mean():.4f} median={benign.median():.4f} | "
          f"anomaly mean={anomaly.mean():.4f} median={anomaly.median():.4f} "
          f"(n_benign={len(benign)}, n_anomaly={len(anomaly)})")
    return out

print("Isolation Forest functions defined.")

{
  "n_estimators": 100,
  "max_samples": "auto",
  "contamination": "auto",
  "random_state": 42,
  "n_jobs": -1
}
Isolation Forest functions defined.


In [4]:
import time

t0 = time.time()
if_cc = fit_isolation_forest(cc_train, CC_FEATURES, IF_CONFIG)
print(f"Fit time: {time.time() - t0:.1f}s")

print("\n=== Isolation Forest — Credit Card scores by split/class ===")
if_cc_scores = {
    "id":  score_split_if(if_cc, cc_id,  CC_FEATURES, "ID"),
    "id2": score_split_if(if_cc, cc_id2, CC_FEATURES, "ID2"),
    "ood": score_split_if(if_cc, cc_ood, CC_FEATURES, "OOD"),
}

Fit time: 0.4s

=== Isolation Forest — Credit Card scores by split/class ===
ID: benign mean=0.4008 median=0.3886 | anomaly mean=0.5857 median=0.6064 (n_benign=21595, n_anomaly=136)
ID2: benign mean=0.4008 median=0.3881 | anomaly mean=0.5753 median=0.5798 (n_benign=21595, n_anomaly=136)
OOD: benign mean=0.4203 median=0.4102 | anomaly mean=0.5718 median=0.5693 (n_benign=139289, n_anomaly=201)


In [5]:
import joblib

IF_CC_DIR = os.path.join(OUTPUT_DIR, "if_credit_card")
os.makedirs(IF_CC_DIR, exist_ok=True)
joblib.dump(if_cc, os.path.join(IF_CC_DIR, "model.joblib"))

if_cc_config = {
    "model": "IsolationForest", "dataset": "credit_card", "hyperparameters": IF_CONFIG,
    "scoring_note": "score_samples() sign-flipped so higher = more anomalous, consistent with AE/VAE convention",
    "scale_note": "IF's score is a bounded path-length measure (~[0,1]); not directly comparable in magnitude to AE/VAE's unbounded reconstruction-error/NLL scores. Use rank correlation or within-model ID-vs-ID2-vs-OOD comparisons, not cross-model raw score magnitude.",
}
with open(os.path.join(IF_CC_DIR, "config.json"), "w") as f:
    json.dump(if_cc_config, f, indent=2)
for split_name, scores_df in if_cc_scores.items():
    scores_df.to_parquet(os.path.join(IF_CC_DIR, f"scores_{split_name}.parquet"), index=False)
print("Saved:", IF_CC_DIR, os.listdir(IF_CC_DIR))

# Scenario T
t0 = time.time()
if_T = fit_isolation_forest(cic_scenarios["scenario_T"]["train"], CIC_FEATURES, IF_CONFIG)
print(f"\nScenario_T fit time: {time.time() - t0:.1f}s")
print("=== Isolation Forest — scenario_T scores by split/class ===")
if_T_scores = {
    "id":  score_split_if(if_T, cic_scenarios["scenario_T"]["id"],  CIC_FEATURES, "ID"),
    "id2": score_split_if(if_T, cic_scenarios["scenario_T"]["id2"], CIC_FEATURES, "ID2"),
    "ood": score_split_if(if_T, cic_scenarios["scenario_T"]["ood"], CIC_FEATURES, "OOD-temporal"),
}

# Scenario A
t0 = time.time()
if_A = fit_isolation_forest(cic_scenarios["scenario_A"]["train"], CIC_FEATURES, IF_CONFIG)
print(f"\nScenario_A fit time: {time.time() - t0:.1f}s")
print("=== Isolation Forest — scenario_A scores by split/class ===")
if_A_scores = {
    "id":  score_split_if(if_A, cic_scenarios["scenario_A"]["id"],  CIC_FEATURES, "ID"),
    "id2": score_split_if(if_A, cic_scenarios["scenario_A"]["id2"], CIC_FEATURES, "ID2"),
    "ood": score_split_if(if_A, cic_scenarios["scenario_A"]["ood"], CIC_FEATURES, "OOD-attack"),
}

Saved: /kaggle/working/models/if_credit_card ['scores_id.parquet', 'scores_ood.parquet', 'scores_id2.parquet', 'model.joblib', 'config.json']

Scenario_T fit time: 1.0s
=== Isolation Forest — scenario_T scores by split/class ===
ID: benign mean=0.3788 median=0.3568 | anomaly mean=0.5805 median=0.6181 (n_benign=195004, n_anomaly=101453)
ID2: benign mean=0.3789 median=0.3573 | anomaly mean=0.5800 median=0.6181 (n_benign=195004, n_anomaly=101453)
OOD-temporal: benign mean=0.3818 median=0.3534 | anomaly mean=0.4447 median=0.3752 (n_benign=795033, n_anomaly=222835)

Scenario_A fit time: 1.1s
=== Isolation Forest — scenario_A scores by split/class ===
ID: benign mean=0.3779 median=0.3554 | anomaly mean=0.5756 median=0.6210 (n_benign=255042, n_anomaly=102542)
ID2: benign mean=0.3778 median=0.3554 | anomaly mean=0.5751 median=0.6209 (n_benign=255042, n_anomaly=102543)
OOD-attack: benign mean=0.3894 median=0.3554 | anomaly mean=0.4332 median=0.3725 (n_benign=394777, n_anomaly=220656)


In [6]:
def category_breakdown(model, ood_df, feature_cols, split_name):
    scores = compute_if_score(model, ood_df, feature_cols)
    anomaly_mask = ood_df["Class"].values == 1
    with_scores = ood_df[anomaly_mask].copy()
    with_scores["if_score"] = scores[anomaly_mask]
    print(f"\n=== IF: Median score by attack category — {split_name} ===")
    print(with_scores.groupby("Label")["if_score"].agg(["median", "count"]).sort_values("median"))

category_breakdown(if_T, cic_scenarios["scenario_T"]["ood"], CIC_FEATURES, "scenario_T (temporal)")
category_breakdown(if_A, cic_scenarios["scenario_A"]["ood"], CIC_FEATURES, "scenario_A (unseen-attack-type)")


=== IF: Median score by attack category — scenario_T (temporal) ===
                              median   count
Label                                       
Bot                         0.360618    1948
PortScan                    0.363566   90694
Web Attack - Sql Injection  0.380264      21
Web Attack - Brute Force    0.396153    1470
Web Attack - XSS            0.396153     652
DDoS                        0.559390  128014
Infiltration                0.631579      36

=== IF: Median score by attack category — scenario_A (unseen-attack-type) ===
            median   count
Label                     
PortScan  0.358090   90694
Bot       0.361588    1948
DDoS      0.527426  128014


In [7]:
IF_T_DIR = os.path.join(OUTPUT_DIR, "if_cic_scenario_T")
os.makedirs(IF_T_DIR, exist_ok=True)
joblib.dump(if_T, os.path.join(IF_T_DIR, "model.joblib"))
if_T_config = {
    "model": "IsolationForest", "dataset": "cic_ids2017", "scenario": "scenario_T (temporal)",
    "hyperparameters": IF_CONFIG,
    "scoring_note": "score_samples() sign-flipped so higher = more anomalous",
    "per_category_finding": {
        "description": "Bot and PortScan near-benign (within ~2-3% of benign median); DDoS clearly detectable. Consistent with AE-T and VAE-T. Web Attacks show more separation under IF than under AE/VAE.",
        "median_by_category": {"Bot": 0.360618, "PortScan": 0.363566, "Web Attack - Sql Injection": 0.380264,
                                "Web Attack - Brute Force": 0.396153, "Web Attack - XSS": 0.396153,
                                "DDoS": 0.559390, "Infiltration": 0.631579},
    },
}
with open(os.path.join(IF_T_DIR, "config.json"), "w") as f:
    json.dump(if_T_config, f, indent=2)
for split_name, scores_df in if_T_scores.items():
    scores_df.to_parquet(os.path.join(IF_T_DIR, f"scores_{split_name}.parquet"), index=False)
print("Saved:", IF_T_DIR, os.listdir(IF_T_DIR))

IF_A_DIR = os.path.join(OUTPUT_DIR, "if_cic_scenario_A")
os.makedirs(IF_A_DIR, exist_ok=True)
joblib.dump(if_A, os.path.join(IF_A_DIR, "model.joblib"))
if_A_config = {
    "model": "IsolationForest", "dataset": "cic_ids2017", "scenario": "scenario_A (unseen-attack-type)",
    "hyperparameters": IF_CONFIG,
    "scoring_note": "score_samples() sign-flipped so higher = more anomalous",
    "per_category_finding": {
        "description": "PortScan and Bot near-indistinguishable from benign (within ~1-2%); DDoS clearly detectable. Matches AE-A and VAE-A exactly — three-for-three model agreement under unseen-attack-type shift.",
        "median_by_category": {"PortScan": 0.358090, "Bot": 0.361588, "DDoS": 0.527426},
    },
}
with open(os.path.join(IF_A_DIR, "config.json"), "w") as f:
    json.dump(if_A_config, f, indent=2)
for split_name, scores_df in if_A_scores.items():
    scores_df.to_parquet(os.path.join(IF_A_DIR, f"scores_{split_name}.parquet"), index=False)
print("Saved:", IF_A_DIR, os.listdir(IF_A_DIR))

Saved: /kaggle/working/models/if_cic_scenario_T ['scores_id.parquet', 'scores_ood.parquet', 'scores_id2.parquet', 'model.joblib', 'config.json']
Saved: /kaggle/working/models/if_cic_scenario_A ['scores_id.parquet', 'scores_ood.parquet', 'scores_id2.parquet', 'model.joblib', 'config.json']


In [8]:
import shutil
shutil.make_archive("/kaggle/working/models_archive_nb3", "zip", "/kaggle/working/models")
print("Size (MB):", os.path.getsize("/kaggle/working/models_archive_nb3.zip") / (1024*1024))

Size (MB): 15.494061470031738
